In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model

# Load CIFAR-10 dataset
(x_train, y_train), (_, _) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0

# Load pretrained VGG16 model
base_model = VGG16(weights='imagenet', include_top=True)

# Create a feature extractor model
feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer('fc1').output)

# Parameters
batch_size = 64
num_batches = int(np.ceil(len(x_train) / batch_size))

# Initialize empty list for features
features_list = []

# Process batches
for batch_idx in range(num_batches):
    start = batch_idx * batch_size
    end = min((batch_idx + 1) * batch_size, len(x_train))
    
    x_batch = x_train[start:end]
    
    # Resize the batch
    x_batch_resized = tf.image.resize(x_batch, (224, 224))
    
    # Extract features
    features_batch = feature_extractor.predict(x_batch_resized, verbose=0)
    
    features_list.append(features_batch)

# Concatenate all features
features = np.vstack(features_list)

# Flatten labels
y_train = y_train.flatten()

# Reduce dimensions
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features)

# Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(features_pca[:, 0], features_pca[:, 1], c=y_train, cmap='tab10', alpha=0.7)
plt.legend(*scatter.legend_elements(), title="Classes")
plt.title('Feature Visualization with PCA (VGG16 Features) - All CIFAR-10 Samples')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.grid(True)
plt.show()


In [8]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# 1. Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=False)

# 2. Load pretrained VGG16 model
vgg16 = torchvision.models.vgg16(pretrained=True)
vgg16.eval()

# 3. Remove the last layer to extract features
feature_extractor = nn.Sequential(*list(vgg16.children())[:-1])  # Remove classifier

# 4. Extract features
all_features = []
all_labels = []

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
feature_extractor = feature_extractor.to(device)

with torch.no_grad():
    for inputs, labels in trainloader:
        inputs = inputs.to(device)
        features = feature_extractor(inputs)
        features = features.view(features.size(0), -1)  # Flatten
        all_features.append(features.cpu())
        all_labels.append(labels)

all_features = torch.cat(all_features).numpy()
all_labels = torch.cat(all_labels).numpy()

# 5. Reduce dimensions

# --- Using PCA
pca = PCA(n_components=2)
features_pca = pca.fit_transform(all_features)

# --- Using t-SNE (optional)
# tsne = TSNE(n_components=2, random_state=42, perplexity=30)
# features_pca = tsne.fit_transform(all_features)

# 6. Plot
plt.figure(figsize=(10, 8))
scatter = plt.scatter(features_pca[:, 0], features_pca[:, 1], c=all_labels, cmap='tab10', alpha=0.7)
plt.legend(*scatter.legend_elements(), title="Classes")
plt.title('Feature Visualization with PCA (VGG16 Features)')
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.grid(True)
plt.show()


100.0%
c:\Users\MBAhmadi\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\MBAhmadi\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\MBAhmadi/.cache\torch\hub\checkpoints\vgg16-397923af.pth


100.0%


KeyboardInterrupt: 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model

# Force TensorFlow to use GPU if available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU mode activated")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ No GPU found, running on CPU.")

# Load CIFAR-10 dataset
(x_train, y_train), (_, _) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0

# Load pretrained VGG16 model
base_model = VGG16(weights='imagenet', include_top=True)

# Create a feature extractor model
feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer('fc1').output)

# Parameters
batch_size = 128  # You can increase if your GPU allows
num_batches = int(np.ceil(len(x_train) / batch_size))

# Initialize empty list for features
features_list = []

# Process batches
for batch_idx in range(num_batches):
    start = batch_idx * batch_size
    end = min((batch_idx + 1) * batch_size, len(x_train))
    
    x_batch = x_train[start:end]
    
    # Resize the batch
    x_batch_resized = tf.image.resize(x_batch, (224, 224))
    
    # Extract features
    features_batch = feature_extractor.predict(x_batch_resized, verbose=0)
    
    features_list.append(features_batch)

# Concatenate all features
features = np.vstack(features_list)

# Flatten labels
y_train = y_train.flatten()

# =========================
# Dimensionality Reduction
# =========================

# 1. PCA
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features)

# 2. t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=1)
features_tsne = tsne.fit_transform(features)

# =========================
# Plotting
# =========================

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Plot PCA
scatter1 = axes[0].scatter(features_pca[:, 0], features_pca[:, 1], c=y_train, cmap='tab10', alpha=0.7)
legend1 = axes[0].legend(*scatter1.legend_elements(), title="Classes")
axes[0].add_artist(legend1)
axes[0].set_title('PCA Visualization (VGG16 Features)')
axes[0].set_xlabel('Component 1')
axes[0].set_ylabel('Component 2')
axes[0].grid(True)

# Plot t-SNE
scatter2 = axes[1].scatter(features_tsne[:, 0], features_tsne[:, 1], c=y_train, cmap='tab10', alpha=0.7)
legend2 = axes[1].legend(*scatter2.legend_elements(), title="Classes")
axes[1].add_artist(legend2)
axes[1].set_title('t-SNE Visualization (VGG16 Features)')
axes[1].set_xlabel('t-SNE Dimension 1')
axes[1].set_ylabel('t-SNE Dimension 2')
axes[1].grid(True)

plt.suptitle('Feature Visualization using PCA and t-SNE', fontsize=20)
plt.tight_layout()
plt.show()


⚠️ No GPU found, running on CPU.


KeyboardInterrupt: 

In [2]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

ModuleNotFoundError: No module named 'tensorflow'

In [11]:
pip uninstall tensorflow

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install tensorflow[and-cuda]

In [1]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


ModuleNotFoundError: No module named 'tensorflow'

In [13]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))


[]


In [3]:
pip install tensorflow-gpu==2.11

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement tensorflow-gpu==2.11 (from versions: 2.12.0)
ERROR: No matching distribution found for tensorflow-gpu==2.11


In [4]:
pip install tensorflow


  Using cached tensorflow-2.19.0-cp312-cp312-win_amd64.whl.metadata (4.1 kB)
Using cached tensorflow-2.19.0-cp312-cp312-win_amd64.whl (376.0 MB)
Note: you may need to restart the kernel to use updated packages.


In [1]:
import tensorflow as tf

# نمایش نسخه‌ی TensorFlow
print("TensorFlow version:", tf.__version__)

# چک کردن GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("✅ GPU Available:", gpus)
else:
    print("❌ No GPU detected.")


TensorFlow version: 2.19.0
❌ No GPU detected.


In [1]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [ ]:
'''import tensorflow as tf
print("TensorFlow version:", tf.__version__)'''

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Model

# Force TensorFlow to use GPU if available
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("✅ GPU mode activated")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ No GPU found, running on CPU.")

# Load CIFAR-10 dataset
(x_train, y_train), (_, _) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0

# Load pretrained VGG16 model
base_model = VGG16(weights='imagenet', include_top=True)

# Create a feature extractor model
feature_extractor = Model(inputs=base_model.input, outputs=base_model.get_layer('fc1').output)

# Parameters
batch_size = 32  # You can increase if your GPU allows
num_batches = int(np.ceil(len(x_train) / batch_size))

# Initialize empty list for features
features_list = []

# Process batches
for batch_idx in range(num_batches):
    start = batch_idx * batch_size
    end = min((batch_idx + 1) * batch_size, len(x_train))
    
    x_batch = x_train[start:end]
    
    # Resize the batch
    x_batch_resized = tf.image.resize(x_batch, (224, 224))
    
    # Extract features
    features_batch = feature_extractor.predict(x_batch_resized, verbose=0)
    
    features_list.append(features_batch)

# Concatenate all features
features = np.vstack(features_list)

# Flatten labels
y_train = y_train.flatten()

# =========================
# Dimensionality Reduction
# =========================

# 1. PCA
pca = PCA(n_components=2)
features_pca = pca.fit_transform(features)

# 2. t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=1)
features_tsne = tsne.fit_transform(features)

# =========================
# Plotting
# =========================

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Plot PCA
scatter1 = axes[0].scatter(features_pca[:, 0], features_pca[:, 1], c=y_train, cmap='tab10', alpha=0.7)
legend1 = axes[0].legend(*scatter1.legend_elements(), title="Classes")
axes[0].add_artist(legend1)
axes[0].set_title('PCA Visualization (VGG16 Features)')
axes[0].set_xlabel('Component 1')
axes[0].set_ylabel('Component 2')
axes[0].grid(True)

# Plot t-SNE
scatter2 = axes[1].scatter(features_tsne[:, 0], features_tsne[:, 1], c=y_train, cmap='tab10', alpha=0.7)
legend2 = axes[1].legend(*scatter2.legend_elements(), title="Classes")
axes[1].add_artist(legend2)
axes[1].set_title('t-SNE Visualization (VGG16 Features)')
axes[1].set_xlabel('t-SNE Dimension 1')
axes[1].set_ylabel('t-SNE Dimension 2')
axes[1].grid(True)

plt.suptitle('Feature Visualization using PCA and t-SNE', fontsize=20)
plt.tight_layout()
plt.show()


✅ GPU mode activated


ResourceExhaustedError: Graph execution error:

Detected at node 'model/block1_conv2/Relu' defined at (most recent call last):
    File "c:\Users\MBAhmadi\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 196, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "c:\Users\MBAhmadi\AppData\Local\Programs\Python\Python310\lib\runpy.py", line 86, in _run_code
      exec(code, run_globals)
    File "C:\Users\MBAhmadi\AppData\Roaming\Python\Python310\site-packages\ipykernel_launcher.py", line 18, in <module>
      app.launch_new_instance()
    File "C:\Users\MBAhmadi\AppData\Roaming\Python\Python310\site-packages\traitlets\config\application.py", line 1075, in launch_instance
      app.start()
    File "C:\Users\MBAhmadi\AppData\Roaming\Python\Python310\site-packages\ipykernel\kernelapp.py", line 739, in start
      self.io_loop.start()
    File "C:\Users\MBAhmadi\AppData\Roaming\Python\Python310\site-packages\tornado\platform\asyncio.py", line 205, in 